# Collecting Weather Data

This notebook collects the needed weather data for the project. It uses open-meteo.com API to get the weather data for the specified location and time period.


We start by installing the necessary libraries from the requirements.txt file and then import them.

In [ ]:
pip install -r requirements.txt

In [ ]:
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

Then we set up the necessary parameters for the API call, such as the latitude and longitude of the location, the start and end dates for the data collection, and the weather parameters we want to collect.

For each electricity zone, we have found an approximate midpoint to collect the weather data. 
For DK1 (western Denmark) we use Silkeborg (55.4426, 11.7901) and for DK2 (eastern Denmark) we use Ringsted (56.1697, 9.5451).

In [ ]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)


locations = [
    {"name": "Ringsted", "latitude": 56.1697, "longitude": 9.5451},
    {"name": "Silkeborg", "latitude": 55.4426, "longitude": 11.7901}]



# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://historical-forecast-api.open-meteo.com/v1/forecast"
params = {
	"latitude": [place["latitude"] for place in locations],
	"longitude": [place["longitude"] for place in locations],
	"start_date": "2022-01-01",
	"end_date": "2024-12-31",
	"hourly": ["wind_speed_10m", "wind_speed_80m", "wind_speed_120m", "wind_speed_180m", "wind_direction_10m", "wind_direction_80m", "wind_direction_120m", "wind_direction_180m", "uv_index", "sunshine_duration"],
}



Then the api is called, the data from the call is stored in a dataframe and the dataframe is saved as a csv file. The csv file is then used in the other notebooks.

In [ ]:

responses = openmeteo.weather_api(url, params = params)

# Process 3 locations
for i, response in enumerate(responses):
	location = locations[i]
	print(f"\nLocation: {location['name']}")
	assert abs(response.Latitude() - location["latitude"]) < 0.01, "Latitude does not match"
	assert abs(response.Longitude() - location["longitude"]) < 0.01, "Longitude does not match"

	print(f"\nCoordinates: {response.Latitude()}°N {response.Longitude()}°E")
	print(f"Elevation: {response.Elevation()} m asl")
	print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
	
	# Process hourly data. The order of variables needs to be the same as requested.
	hourly = response.Hourly()
	hourly_wind_speed_10m = hourly.Variables(0).ValuesAsNumpy()
	hourly_wind_speed_80m = hourly.Variables(1).ValuesAsNumpy()
	hourly_wind_speed_120m = hourly.Variables(2).ValuesAsNumpy()
	hourly_wind_speed_180m = hourly.Variables(3).ValuesAsNumpy()
	hourly_wind_direction_10m = hourly.Variables(4).ValuesAsNumpy()
	hourly_wind_direction_80m = hourly.Variables(5).ValuesAsNumpy()
	hourly_wind_direction_120m = hourly.Variables(6).ValuesAsNumpy()
	hourly_wind_direction_180m = hourly.Variables(7).ValuesAsNumpy()
	hourly_uv_index = hourly.Variables(8).ValuesAsNumpy()
	hourly_sunshine_duration = hourly.Variables(9).ValuesAsNumpy()
	
	hourly_data = {"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)}
	
	hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
	hourly_data["wind_speed_80m"] = hourly_wind_speed_80m
	hourly_data["wind_speed_120m"] = hourly_wind_speed_120m
	hourly_data["wind_speed_180m"] = hourly_wind_speed_180m
	hourly_data["wind_direction_10m"] = hourly_wind_direction_10m
	hourly_data["wind_direction_80m"] = hourly_wind_direction_80m
	hourly_data["wind_direction_120m"] = hourly_wind_direction_120m
	hourly_data["wind_direction_180m"] = hourly_wind_direction_180m
	hourly_data["uv_index"] = hourly_uv_index
	hourly_data["sunshine_duration"] = hourly_sunshine_duration
	
	hourly_dataframe = pd.DataFrame(data = hourly_data)
	print("\nHourly data\n", hourly_dataframe)
	hourly_dataframe.to_csv(f"hourly_data_{location['name']}_{response.Latitude()}_{response.Longitude()}.csv", index = False)
	

# Collecting Demand Data
Our demand data is collected from a kaggle dataset: https://www.kaggle.com/datasets/csafrit2/steel-industry-energy-consumption

This is a dataset of energy consumption for a steel factory called DAEWOO Steel Co. Ltd in Gwangyang, South Korea. The dataset contains energy consumption data from January 1, 2018 to December 31, 2018 for every quarter-hour.


We use the kagglehub library to download the latest version of the dataset and save it to the data directory. The path to the dataset files is printed for reference.

In [ ]:
import kagglehub

# Download latest version to this directory
path = kagglehub.dataset_download("csafrit2/steel-industry-energy-consumption", output_dir="data")



Now we load the dataset into a dataframe and do some basic preprocessing to get it ready for analysis. We convert the date column to a datetime format and extract the week, day of the week, and hour of the day as new columns in the dataframe. This will allow us to analyze the demand patterns based on time.

In [ ]:
import pandas as pd

# Load the dataset into a dataframe
demand_df = pd.read_csv(r"data\Steel_industry_data.csv")

demand_df['date'] = pd.to_datetime(demand_df['date'], format="%d/%m/%Y %H:%M")

# Set the date column as the index
demand_df.set_index('date', inplace=True)


# reduce to hourly data by taking the sum of every 4 rows (since the original data is in 15-minute intervals)
demand_df = demand_df.resample('h').sum()

demand_df.to_csv("steel_industry_hourly_usage.csv", index = True)
